In [2]:
import os
os.chdir("/home/jovyan/work/repo_conteo_gallinas")
os.environ["CUDA_VISIBLE_DEVICES"] = "0, 2"

import torch
from ultralytics import YOLO
import pandas as pd
import shutil
import gc

import optuna


In [5]:
# MODEL_EPOCHS = 50
# MODEL_BATCH = 8
# MODEL_LR = 0.01




In [6]:
# evaluator = BinaryClassificationEvaluator(
#     labelCol="label",
#     rawPredictionCol="probability",
#     metricName="areaUnderROC"
# )


In [7]:
def objective(trial):

    model = YOLO("model/yolo26m.pt")

    # Eliminar caarpetas de resultados
    if os.path.exists("runs"):
        shutil.rmtree("runs")
    
    params = {
        "MODEL_EPOCHS": trial.suggest_int("MODEL_EPOCHS", 100, 300),
        "MODEL_BATCH": trial.suggest_int("MODEL_BATCH", 8, 12),
        "MODEL_LR": trial.suggest_float("MODEL_LR", 0.001, 0.1),
        "MODEL_OPTIMIZER": trial.suggest_categorical("MODEL_OPTIMIZER",
                            ["SGD", "auto"]
                        )
    }

    results = model.train(
        data="artifacts/full_dataset.yaml",
        imgsz=640,
        epochs=params["MODEL_EPOCHS"],
        batch=params["MODEL_BATCH"],
        lr0=params["MODEL_LR"],
        optimizer=params["MODEL_OPTIMIZER"],
        seed=123,
        deterministic=True,
        # device=0,
        device=[0, 1],
        workers=0,
        project='.',
        name='.'
        # device=[0, 1],
        # workers=0,
        # project='.',
        # name='.'
    )
        
    # Lectura resultados
    df = pd.read_csv("runs/detect/results.csv")
    # epoca_final = int(df["epoch"].max())
    # tiempo_segundos = float(df["time"].max())
    # tiempo_min = tiempo_segundos/60
    # tiempo_hora = tiempo_min/60
    
    best = df.loc[df["metrics/mAP50-95(B)"].idxmax()]
    # map50_final = float(best["metrics/mAP50(B)"])
    map5095_final = float(best["metrics/mAP50-95(B)"])
    # precision_final = float(best["metrics/precision(B)"])
    # recall_final = float(best["metrics/recall(B)"])

    # # Número de imágenes
    # imagenes = len(os.listdir("data/deteccion/imgs"))
    # imagenes_train = len(os.listdir("data/dataset/train/images"))
    # imagenes_val = len(os.listdir("data/dataset/val/images"))

    # # Información sonsolidada
    # data_list = {
    #     'epocas_inicial': params["MODEL_EPOCHS"]
    #     , 'epocas_fin': epoca_final
    #     , 'batch': params["MODEL_BATCH"]
    #     , 'learning_rate': params["MODEL_LR"]
    #     , 'optimizer': "SGD"
    #     , 'total_imagenes': imagenes
    #     , 'total_imagenes_train': imagenes_train
    #     , 'total_imagenes_val': imagenes_val

    #     , 'tiempo_minutos': tiempo_min
    #     , 'tiempo_horas': tiempo_hora
    #     , 'map50': map50_final
    #     , 'map50_95': map5095_final
    #     , "precision": precision_final
    #     , "recall": recall_final
    # }
    # data_results = pd.DataFrame([data_list])
    # data_results.to_csv(f"tmp/results_opt/result_{trial.number}.csv", index = False, encoding = "utf-8")

    del model
    gc.collect()
    torch.cuda.empty_cache()

    return map5095_final


In [8]:
study = optuna.create_study(
    direction="maximize",
    study_name="yolo_detect",
    sampler=optuna.samplers.TPESampler(seed=123)
)

study.optimize(
    objective,
    n_trials=72
)

df_results = study.trials_dataframe()
df_results.to_csv(
    "tmp/yolo26_ft_opt_11.csv",
    index=False
)



[I 2026-11-11 05:49:11,970] A new study created in memory with name: yolo_detect


Ultralytics 8.4.96 🚀 Python-3.13.11 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 48640MiB)
                                                       CUDA:1 (NVIDIA RTX 6000 Ada Generation, 48632MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=9, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=artifacts/full_dataset.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=239, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.02345829390285611, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=mode

[I 2026-11-11 05:59:12,125] Trial 0 finished with value: 0.2425 and parameters: {'MODEL_EPOCHS': 239, 'MODEL_BATCH': 9, 'MODEL_LR': 0.02345829390285611, 'MODEL_OPTIMIZER': 'auto'}. Best is trial 0 with value: 0.2425.


Ultralytics 8.4.96 🚀 Python-3.13.11 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 48640MiB)
                                                       CUDA:1 (NVIDIA RTX 6000 Ada Generation, 48632MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=12, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=artifacts/full_dataset.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=185, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.06879814411990147, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=mod

[I 2026-11-11 06:09:03,595] Trial 1 finished with value: 0.22276 and parameters: {'MODEL_EPOCHS': 185, 'MODEL_BATCH': 12, 'MODEL_LR': 0.06879814411990147, 'MODEL_OPTIMIZER': 'SGD'}. Best is trial 0 with value: 0.2425.


Ultralytics 8.4.96 🚀 Python-3.13.11 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 48640MiB)
                                                       CUDA:1 (NVIDIA RTX 6000 Ada Generation, 48632MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=11, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=artifacts/full_dataset.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=168, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.04441865222328282, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=mod

... (43 outputs left out)

Ultralytics 8.4.96 🚀 Python-3.13.11 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 48640MiB)
                                                       CUDA:1 (NVIDIA RTX 6000 Ada Generation, 48632MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=12, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=artifacts/full_dataset.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=261, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.012079832013907242, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=mo

[I 2026-11-11 16:48:50,698] Trial 70 finished with value: 0.23986 and parameters: {'MODEL_EPOCHS': 261, 'MODEL_BATCH': 12, 'MODEL_LR': 0.012079832013907242, 'MODEL_OPTIMIZER': 'auto'}. Best is trial 41 with value: 0.25962.


Ultralytics 8.4.96 🚀 Python-3.13.11 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 48640MiB)
                                                       CUDA:1 (NVIDIA RTX 6000 Ada Generation, 48632MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=11, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=artifacts/full_dataset.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=279, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0038949310636884516, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=m

[I 2026-11-11 16:57:59,559] Trial 71 finished with value: 0.25311 and parameters: {'MODEL_EPOCHS': 279, 'MODEL_BATCH': 11, 'MODEL_LR': 0.0038949310636884516, 'MODEL_OPTIMIZER': 'SGD'}. Best is trial 41 with value: 0.25962.


In [9]:
df_results

,number,value,datetime_start,datetime_complete,duration,params_MODEL_BATCH,params_MODEL_EPOCHS,params_MODEL_LR,params_MODEL_OPTIMIZER,state
0,0,0.24250,2026-11-11 05:49:11.972113,2026-11-11 05:59:12.124929,0 days 00:10:00.152816,9,239,0.023458,auto,COMPLETE
1,1,0.22276,2026-11-11 05:59:12.126324,2026-11-11 06:09:03.595797,0 days 00:09:51.469473,12,185,0.068798,SGD,COMPLETE
2,2,0.24339,2026-11-11 06:09:03.596955,2026-11-11 06:17:51.243851,0 days 00:08:47.646896,11,168,0.044419,auto,COMPLETE
3,3,0.24097,2026-11-11 06:17:51.245434,2026-11-11 06:27:21.171881,0 days 00:09:29.926447,8,248,0.018370,auto,COMPLETE
4,4,0.24055,2026-11-11 06:27:21.173453,2026-11-11 06:36:24.767349,0 days 00:09:03.593896,12,227,0.072721,auto,COMPLETE
...,...,...,...,...,...,...,...,...,...,...
67,67,0.25817,2026-11-11 16:10:24.358986,2026-11-11 16:19:34.614867,0 days 00:09:10.255881,11,266,0.003427,SGD,COMPLETE
68,68,0.25533,2026-11-11 16:19:34.616434,2026-11-11 16:32:51.412211,0 days 00:13:16.795777,10,266,0.001011,SGD,COMPLETE
69,69,0.25051,2026-11-11 16:32:51.413292,2026-11-11 16:40:51.405730,0 days 00:07:59.992438,11,286,0.017452,SGD,COMPLETE
70,70,0.23986,2026-11-11 16:40:51.407392,2026-11-11 16:48:50.698100,0 days 00:07:59.290708,12,261,0.012080,auto,COMPLETE
